In [ ]:
import tensorflow as tf
print('Tensorflow version:',tf.__version__)

## Load the MNIST dataset
`x_train`/`x_test` represents pixel values in the range from 0-255 (scaled to a 0-1) range.
`y_train`/`y_test` represents the number depicted by the pixel values.




In [ ]:
mnist = tf.keras.datasets.mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

## Build the machine learning model

In [ ]:
model = tf.keras.models.Sequential(
    [
        tf.keras.layers.Flatten(input_shape=(28,28)),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(10)
    ]
)
predictions = model(x_train[:1]).numpy()
tf.nn.softmax(predictions).numpy()

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
loss_fn(y_train[:1], predictions).numpy()

model.compile(
  optimizer='adam',
  loss=loss_fn,
  metrics=['accuracy']
)

## Tain and evaluate the model

In [ ]:
model.fit(x_train, y_train, epochs=5)
model.evaluate(x_test,  y_test, verbose=2)

## Predict using the model

In [ ]:
probability_model = tf.keras.Sequential([
  model,
  tf.keras.layers.Softmax()
])
predictions = probability_model(x_test[6:7]).numpy()[0]

for i, prob in enumerate(predictions):
    print(f"Number {i}: {prob:.2%}")

In [ ]:
import numpy as np
from skimage import exposure
import base64
import requests
from PIL import Image, ImageOps, ImageChops
from io import BytesIO

def replace_transparent_background(image):
  if image.mode != 'RGBA':
    return image

  image_arr = np.array(image)

  if len(image_arr.shape) == 2:
    return image

  alpha1 = 0
  r2, g2, b2, alpha2 = 255, 255, 255, 255

  red, green, blue, alpha = image_arr[:, :, 0], image_arr[:, :, 1], image_arr[:, :, 2], image_arr[:, :, 3]
  mask = (alpha == alpha1)
  image_arr[:, :, :4][mask] = [r2, g2, b2, alpha2]

  return Image.fromarray(image_arr)


def trim_borders(image):
  bg = Image.new(image.mode, image.size, image.getpixel((0,0)))
  diff = ImageChops.difference(image, bg)
  diff = ImageChops.add(diff, diff, 2.0, -100)
  bbox = diff.getbbox()
  if bbox:
      return image.crop(bbox)

  return image


def pad_image(image):
  return ImageOps.expand(image, border=30, fill='#fff')


def to_grayscale(image):
  return image.convert('L')


def invert_colors(image):
    return ImageOps.invert(image)


def resize_image(image):
  return image.resize((28, 28), Image.Resampling.BILINEAR)


def process_image(image):
  steps = [
    replace_transparent_background,
    trim_borders,
    pad_image,
    to_grayscale,
    invert_colors,
    resize_image
  ]

  for step in steps:
      image = step(image)

  return image

In [ ]:
# @title Default title text
from IPython.display import display

url = 'https://e7.pngegg.com/pngimages/133/35/png-clipart-handwriting-letter-others-miscellaneous-text-thumbnail.png' #@param {type:"string"}
image = Image.open(BytesIO(requests.get(url).content))
processed_image = process_image(image)

display(processed_image)

In [ ]:
pixels = np.array(processed_image)
pixels_normalized = pixels / 255.0
pixels_reshaped = pixels_normalized.reshape(1, 28, 28)

predictions = probability_model(pixels_reshaped).numpy()[0]

for i, prob in enumerate(predictions):
    print(f"Number {i}: {prob:.2%}")

## Save model for use with `tensorflowjs`

In [ ]:
# Convert to TensorFlow.js Graph model format (better compatibility)
# IMPORTANT: Run ALL cells above first
%pip install tensorflowjs -q

# Save as SavedModel format first
probability_model.export('saved_model')

# Convert to TensorFlow.js Graph model using CLI
!tensorflowjs_converter --input_format=tf_saved_model --output_format=tfjs_graph_model saved_model tfjs_model

# Download the converted model
!zip -r tfjs_model.zip tfjs_model
from google.colab import files
files.download('tfjs_model.zip')